In [1]:
import sys
from dataclasses import dataclass, field
from itertools import product
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from IPython.display import display
from matplotlib.colors import Normalize
from tqdm.auto import tqdm

project_root = Path.cwd()
if not (project_root / "tasks").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    import scienceplots  # noqa: F401
except ModuleNotFoundError:
    pass
else:
    plt.style.use("science")

from network_creation.plotting.plot_pair import plot_pair
from network_creation.plotting.reference_values import (
    get_observer_reference_style,
    get_sigmoid_fit_reference_values,
    positive_reference_norm,
)
from network_creation.stats.sigmoid_fit import fit_zero_start_sigmoids
from tasks.policies import load_policy_results


In [2]:
%matplotlib widget


In [3]:
LAST_STEP = "Last step"
SIGMOID_VALUES = {
    "Final plateau (A)": ("A", "estimated final plateau"),
    "Approach speed (k)": ("k", "estimated approach speed"),
    "Time shift (t0)": ("t0", "estimated time shift"),
}


@dataclass(frozen=True)
class PlotRequest:
    observable: str | None = None
    n_agents: int | None = None
    proportion: float | None = None
    value: str | None = None


@dataclass(frozen=True)
class PlaygroundConfig:
    result_store: str | Path | None = None
    final_observables: tuple[str, ...] | None = None
    fit_observables: tuple[str, ...] = (
        "centered_mean_welfare.mean",
        "central_point_of_dominance.mean",
    )
    plot: PlotRequest = field(default_factory=PlotRequest)
    method: str = "gaussian"
    resolution: int = 50
    point_size: float = 3
    pair_atol: float = 1e-5

config = PlaygroundConfig(result_store=project_root / "artifacts" / "simulation_runs" / "run-20260430T144413.zarr")


In [4]:
@dataclass(frozen=True)
class ResultContext:
    project_root: Path
    result_store_path: Path
    ds: xr.Dataset
    proportion_dim: str
    theta_labels: tuple[str, ...]
    observable_labels: tuple[str, ...]
    steps: np.ndarray


@dataclass(frozen=True)
class PlotData:
    observable: str
    n_agents: int
    proportion: float
    value: str
    theta: np.ndarray
    y: np.ndarray
    references: dict[str, float]
    norm: Normalize
    title: str
    colorbar_label: str
    n_pair_axes: int


def resolve_result_store(root, value):
    if value is not None:
        path = Path(value).expanduser()
        return path if path.is_absolute() else root / path
    candidates = sorted(root.glob("artifacts/**/*.zarr"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError("No .zarr result stores found under artifacts/")
    return candidates[0]


def load_context(config, root):
    ds = load_policy_results(resolve_result_store(root, config.result_store))
    proportion_dims = tuple(dim for dim in ds["x"].dims if dim.endswith("_proportion"))
    if len(proportion_dims) != 1:
        raise ValueError(f"Expected one policy-proportion dimension, found {proportion_dims}")
    return ResultContext(
        project_root=root,
        result_store_path=Path(ds.attrs["result_store_path"]),
        ds=ds,
        proportion_dim=proportion_dims[0],
        theta_labels=tuple(str(v) for v in ds.coords["theta_parameter"].values),
        observable_labels=tuple(str(v) for v in ds.coords["observable"].values),
        steps=np.asarray(ds.coords["step"].values, dtype=float),
    )


def configured_observables(ctx, requested, *, strict):
    if requested is None:
        return ctx.observable_labels
    selected = tuple(str(value) for value in requested)
    missing = tuple(value for value in selected if value not in ctx.observable_labels)
    if missing and strict:
        raise ValueError(f"Unknown observables: {missing}. Available: {ctx.observable_labels}")
    if missing:
        print(f"Skipping missing observables: {missing}")
    return tuple(value for value in selected if value in ctx.observable_labels)


def empty_observer_values(ctx):
    value_labels = (LAST_STEP, *SIGMOID_VALUES)
    dims = ("observable", "n_agents", ctx.proportion_dim, "value", "parameter_row")
    shape = (
        ctx.ds.sizes["observable"],
        ctx.ds.sizes["n_agents"],
        ctx.ds.sizes[ctx.proportion_dim],
        len(value_labels),
        ctx.ds.sizes["parameter_row"],
    )
    coords = {dim: ctx.ds.coords[dim] for dim in dims if dim != "value"}
    coords["value"] = ("value", list(value_labels))
    return xr.DataArray(
        np.full(shape, np.nan, dtype=np.float32),
        dims=dims,
        coords=coords,
        name="observer_values",
    )


def build_last_step_values(ctx, config, values):
    observables = configured_observables(ctx, config.final_observables, strict=True)
    tasks = list(product(observables, ctx.ds.coords["n_agents"].values, ctx.ds.coords[ctx.proportion_dim].values))
    final = ctx.ds["x"].isel(step=-1)
    for observable, n_agents, proportion in tqdm(tasks, desc="Last-step values", unit="map"):
        values.loc[dict(observable=observable, n_agents=n_agents, **{ctx.proportion_dim: proportion}, value=LAST_STEP)] = (
            final.sel(observable=observable, n_agents=n_agents, **{ctx.proportion_dim: proportion}).to_numpy()
        )


def build_sigmoid_values(ctx, config, values):
    observables = configured_observables(ctx, config.fit_observables, strict=False)
    tasks = list(product(observables, ctx.ds.coords["n_agents"].values, ctx.ds.coords[ctx.proportion_dim].values))
    for observable, n_agents, proportion in tqdm(tasks, desc="Sigmoid fits", unit="fit"):
        selector = {"observable": observable, "n_agents": n_agents, ctx.proportion_dim: proportion}
        Y = ctx.ds["x"].sel(selector).transpose("parameter_row", "step").to_numpy()
        params, _ = fit_zero_start_sigmoids(Y, t=ctx.steps)
        params = np.asarray(params, dtype=np.float32)
        for col, value_label in enumerate(SIGMOID_VALUES):
            values.loc[dict(observable=observable, n_agents=n_agents, **{ctx.proportion_dim: proportion}, value=value_label)] = params[:, col]


def build_observer_values(ctx, config):
    values = empty_observer_values(ctx)
    build_last_step_values(ctx, config, values)
    build_sigmoid_values(ctx, config, values)
    return values


In [5]:
def pair_axes_count(theta, atol):
    return int(np.count_nonzero(np.var(theta, axis=0) > atol) - 1)


def finite_mask(theta, y):
    return np.isfinite(theta).all(axis=1) & np.isfinite(y)


def resolved_norm(values, references=(), norm=None):
    if norm is not None:
        return norm
    finite = np.asarray([*np.asarray(values, dtype=float).ravel(), *references], dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return Normalize(vmin=0.0, vmax=1.0)
    return Normalize(vmin=float(finite.min()), vmax=float(finite.max()))


def candidate_keys(ctx, values, request):
    for observable, n_agents, proportion, value in product(
        values.coords["observable"].values,
        values.coords["n_agents"].values,
        values.coords[ctx.proportion_dim].values,
        values.coords["value"].values,
    ):
        if request.observable is not None and str(observable) != request.observable:
            continue
        if request.n_agents is not None and int(n_agents) != int(request.n_agents):
            continue
        if request.proportion is not None and float(proportion) != float(request.proportion):
            continue
        if request.value is not None and str(value) != request.value:
            continue
        yield str(observable), int(n_agents), float(proportion), str(value)


def references_and_norm(ctx, observable, n_agents, value, theta, y):
    if value == LAST_STEP:
        references, _ = get_observer_reference_style(
            observable,
            y,
            n_agents=n_agents,
            theta_labels=ctx.theta_labels,
            theta_values=theta,
        )
        return references, resolved_norm(y, references.values()), observable


    reference_key, label = SIGMOID_VALUES[value]
    sigmoid_refs = get_sigmoid_fit_reference_values(
        observable,
        n_agents=n_agents,
    )
    references = sigmoid_refs.get(reference_key, {})
    if value == "Final plateau (A)":
        _, norm = get_observer_reference_style(
            observable,
            y,
            n_agents=n_agents,
            theta_labels=ctx.theta_labels,
            theta_values=theta,
        )
        return references, resolved_norm(y, references.values(), norm), label
    return references, positive_reference_norm(y, references), label


def select_plot_data(ctx, values, request, config):
    theta_data = ctx.ds["theta"]
    for observable, n_agents, proportion, value in candidate_keys(ctx, values, request):
        selector = {"observable": observable, "n_agents": n_agents, ctx.proportion_dim: proportion, "value": value}
        y = values.sel(selector).to_numpy()
        theta = theta_data.sel(n_agents=n_agents, **{ctx.proportion_dim: proportion}).to_numpy()
        mask = finite_mask(theta, y)
        if not mask.any():
            continue
        n_pair_axes = pair_axes_count(theta[mask], config.pair_atol)
        if n_pair_axes < 1:
            continue
        references, norm, colorbar_label = references_and_norm(ctx, observable, n_agents, value, theta[mask], y[mask])
        return PlotData(
            observable=observable,
            n_agents=n_agents,
            proportion=proportion,
            value=value,
            theta=theta[mask],
            y=y[mask],
            references=references,
            norm=norm,
            title=value,
            colorbar_label=colorbar_label,
            n_pair_axes=n_pair_axes,
        )
    raise ValueError(f"No finite observer values match {request}")


In [6]:
def plot_observer_map(ctx, data, config, *, display_plot=True):
    n = data.n_pair_axes
    if n == 1:
        fig = plt.figure(figsize=(7.0, 5.0))
        grid = fig.add_gridspec(1, 1, left=0.08, right=0.72, bottom=0.12, top=0.90)
        axes = np.empty((1, 1), dtype=object)
        axes[0, 0] = fig.add_subplot(grid[0, 0])
        colorbar_ax = fig.add_axes([0.82, 0.12, 0.025, 0.78])
    else:
        fig, axes = plt.subplots(n, n, figsize=(5.0 * n, 5.0 * n), layout="constrained", squeeze=False)

    plot_pair(
        X=data.theta,
        y=data.y,
        labels=ctx.theta_labels,
        atol=config.pair_atol,
        cmap="viridis",
        method=config.method,
        resolution=config.resolution,
        point_size=config.point_size,
        norm=data.norm,
        fig=fig,
        axes=axes,
        ordered_discrete_keys=[key for key in ("visibility_distance",) if key in ctx.theta_labels],
    )

    if n > 1:
        colorbar_slot_ax = axes[0, -1]
        colorbar_slot_ax.clear()
        colorbar_slot_ax.axis("off")
        colorbar_ax = colorbar_slot_ax.inset_axes([0.42, 0.0, 0.08, 1.0])

    colorbar = fig.colorbar(
        plt.cm.ScalarMappable(cmap="viridis", norm=data.norm),
        cax=colorbar_ax,
        format="%.2f",
    )
    colorbar.set_label(data.colorbar_label)

    references = sorted(
        (
            (str(name), float(value))
            for name, value in data.references.items()
            if np.isfinite(value)
        ),
        key=lambda item: item[1],
    )
    if references:
        colorbar_ax.tick_params(
            axis="y",
            which="both",
            left=False,
            right=False,
            labelleft=False,
            labelright=False,
        )
    colorbar_ax.yaxis.set_label_position("right")
    colorbar_ax.yaxis.labelpad = 34

    for name, value in references:
        name_text = colorbar_ax.text(
            -0.35,
            value,
            name,
            transform=colorbar_ax.get_yaxis_transform(),
            ha="right",
            va="center",
            fontsize="small",
            clip_on=False,
        )
        value_text = colorbar_ax.text(
            1.35,
            value,
            f"{value:.2f}",
            transform=colorbar_ax.get_yaxis_transform(),
            ha="left",
            va="center",
            fontsize="small",
            clip_on=False,
        )
        name_text.set_in_layout(False)
        value_text.set_in_layout(False)

    fig.suptitle(
        "\n".join(
            [
                str(data.title),
                f"n_agents={data.n_agents}, "
                f"{ctx.proportion_dim}={data.proportion:.3g}, "
                f"{data.observable}",
            ]
        )
    )
    if display_plot:
        display(fig.canvas)
    return fig, axes, colorbar_ax


In [7]:
ctx = load_context(config, project_root)
observer_values = build_observer_values(ctx, config)

print(f"Loaded {ctx.result_store_path.relative_to(ctx.project_root)}")

Last-step values:   0%|          | 0/88 [00:00<?, ?map/s]

Sigmoid fits:   0%|          | 0/44 [00:00<?, ?fit/s]

Loaded artifacts/simulation_runs/run-20260430T144413.zarr


In [ ]:
def available_plot_keys(ctx, values):
    has_data = values.notnull().any("parameter_row")
    keys = []
    for observable, n_agents, proportion, value in product(
        values.coords["observable"].values,
        values.coords["n_agents"].values,
        values.coords[ctx.proportion_dim].values,
        values.coords["value"].values,
    ):
        selector = {
            "observable": observable,
            "n_agents": n_agents,
            ctx.proportion_dim: proportion,
            "value": value,
        }
        if bool(has_data.sel(selector).item()):
            keys.append((str(observable), int(n_agents), float(proportion), str(value)))
    return keys


def proportion_label(name):
    return f"{name.removesuffix('_proportion').replace('_', ' ')} share"


def labeled(title, widget):
    return widgets.VBox([widgets.HTML(f"<b>{title}</b>"), widget])


def make_plot_widget(ctx, values, config):
    keys = available_plot_keys(ctx, values)
    if not keys:
        raise ValueError("No observer values are available to plot.")

    def matches(observable=None, n_agents=None, proportion=None):
        return [
            key
            for key in keys
            if (observable is None or key[0] == observable)
            and (n_agents is None or key[1] == n_agents)
            and (proportion is None or key[2] == proportion)
        ]

    def unique(index, rows):
        return list(dict.fromkeys(row[index] for row in rows))

    observer = widgets.Select(rows=8, layout=widgets.Layout(width="320px", height="170px"))
    n_agents = widgets.Select(rows=5, layout=widgets.Layout(width="90px", height="120px"))
    proportion = widgets.Select(rows=5, layout=widgets.Layout(width="170px", height="120px"))
    value = widgets.Select(rows=5, layout=widgets.Layout(width="230px", height="120px"))
    output = widgets.Output()

    observer.options = unique(0, keys)

    state = {"updating": False, "fig": None}

    def set_options(control, options):
        control.options = options
        valid_values = [item[1] if isinstance(item, tuple) else item for item in options]
        if control.value not in valid_values:
            control.value = valid_values[0]

    def refresh_options(*_):
        if state["updating"]:
            return

        state["updating"] = True
        try:
            n_opts = unique(1, matches(observer.value))
            set_options(n_agents, n_opts)

            p_opts = unique(2, matches(observer.value, n_agents.value))
            set_options(proportion, [(f"{p:.2g}", p) for p in p_opts])

            v_opts = unique(3, matches(observer.value, n_agents.value, proportion.value))
            set_options(value, v_opts)
        finally:
            state["updating"] = False

        draw()

    def draw(*_):
        if state["updating"]:
            return

        if state["fig"] is not None:
            plt.close(state["fig"])
            state["fig"] = None

        data = select_plot_data(
            ctx,
            values,
            PlotRequest(observer.value, n_agents.value, proportion.value, value.value),
            config,
        )
        fig, _, _ = plot_observer_map(ctx, data, config, display_plot=False)
        state["fig"] = fig

        with output:
            output.clear_output(wait=True)
            plt.show(fig)


    for control in (observer, n_agents, proportion):
        control.observe(refresh_options, names="value")

    value.observe(draw, names="value")

    refresh_options()

    controls = widgets.HBox(
        [
            labeled("Observer", observer),
            labeled("n agents", n_agents),
            labeled(proportion_label(ctx.proportion_dim), proportion),
            labeled("Value", value),
        ],
        layout=widgets.Layout(gap="18px", align_items="flex-start", flex_flow="row wrap"),
    )
    return widgets.VBox([controls, output])


display(make_plot_widget(ctx, observer_values, config))
